In [2]:
from dotenv import load_dotenv
load_dotenv()


True

In [3]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [ ]:
def llm(prompt, model="openai/gpt-oss-20b"):
    response = openai_client.responses.create(
        #model='llama-3.3-70b-versatile',
        model = model,
        input=prompt
    )
    return response.output_text

In [5]:
question = 'I just discovered the course. Can I join now?'
answer = llm(question)
print(answer)

Hey there! 🎉

Absolutely, you can still join—just depends on a few quick details:

| What to Check | Why It Matters | What Happens Next |
|----------------|----------------|--------------------|
| **Course Term & Session** | Some courses run in fixed sessions (e.g., Fall 2025, Summer 2026). If you missed the registration window, you might still be able to add it via *Course Add/Drop* (or waitlist if the class is full). | If it’s the current session, you can add it right away. If the session is over, you can enroll in the next available session. |
| **Prerequisites** | Many courses have pre‑reqs (e.g., a foundational course or a certain GPA). | If you’re missing a pre‑req, you might still enroll in a *waitlist* or *pre‑approval* route. |
| **Capacity / Waitlist** | Classes often have a max size. If it’s full, you can join a waitlist. | If you get a spot, you’ll be automatically enrolled. |
| **Financial Hold / Fees** | Outstanding fees or holds can block enrollment. | Resolve any holds 

In [6]:
import requests
import os

api_key = os.environ.get("GROQ_API_KEY")
url = "https://api.groq.com/openai/v1/models"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

response = requests.get(url, headers=headers)

print(response.json())


{'object': 'list', 'data': [{'model_card': 'https://huggingface.co/openai/whisper-large-v3', 'id': 'whisper-large-v3', 'object': 'model', 'created': 1693721698, 'owned_by': 'OpenAI', 'active': True, 'context_window': 448, 'public_apps': None, 'max_completion_tokens': 448, 'hugging_face_id': 'openai/whisper-large-v3', 'name': 'Whisper', 'input_modalities': ['audio'], 'output_modalities': ['transcription'], 'context_length': 448, 'max_output_length': 448}, {'id': 'meta-llama/llama-prompt-guard-2-22m', 'object': 'model', 'created': 1748632101, 'owned_by': 'Meta', 'active': True, 'context_window': 512, 'public_apps': None, 'max_completion_tokens': 512, 'name': 'Llama Prompt Guard 2 22M', 'input_modalities': ['text'], 'output_modalities': ['text'], 'context_length': 512, 'max_output_length': 512, 'pricing': {'prompt': '0.00000003', 'completion': '0.00000003', 'image': '0', 'request': '0', 'input_cache_read': '0.000000015'}, 'supported_sampling_parameters': ['temperature', 'top_p', 'stop', '

In [7]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [8]:
courses_raw

[{'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 482},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 161},
 {'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 398},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 73},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 249}]

In [9]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1456

In [10]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [11]:
question = 'I just discovered the course. Can I join now?'
search_result = index.search(question, 
filter_dict={"course": "llm-zoomcamp"},
num_results=3,
boost_dict={'question': 2, 'section': 0.5})
search_result


[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not 

In [12]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {'question': 2, 'section': 0.5}
    filter_dict = {"course": course}
    return index.search(question, 
        filter_dict=filter_dict,
        num_results=3,
        boost_dict=boost_dict
        )


---

In [13]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [14]:
USER_PROMPT_TEMPLATE = '''
QUESTION: 
{question}

CONTEXT:
{context}
'''


In [15]:
def build_context(search_result):
    lines = []

    for doc in search_result:
        lines.append(doc['section'])
        lines.append('Q: ' + doc['question'])
        lines.append('A: ' + doc['answer'])
        lines.append('')

    return '\n'.join(lines).strip()



In [16]:
context = build_context(search_result)
print(context)

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Does the course certificate show the number of course hours?
A: No. The certificate does not state a total number of hours.


In [19]:
def build_prompt(question, search_result):
    context = build_context(search_result)
    return USER_PROMPT_TEMPLATE.format(question=question, context=context)


In [ ]:
prompt = build_prompt(question, search_result)
print(prompt)




QUESTION: 
I just discovered the course. Can I join now?

CONTEXT:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Does the course certificate show the number of course hours?
A: No. The certificate does not state a total number of hours.



---

In [21]:
response = llm(prompt)
print(response)

Absolutely – you can join the course even if you’ve just found it!  
- **Enrollment**: Just start working through the material and submitting the assignments while the submission window is still open.  
- **Certificate**: To receive a certificate, you’ll need to submit your final project before the course closes its submission period.  

Feel free to dive in right away; no registration confirmation is required to begin learning and submitting homework.


In [22]:
message_history = [
    {"role": 'developer', 'content': INSTRUCTIONS},
    {"role": 'user', 'content': prompt}
]

response = llm(message_history)
print(response)


Yes—feel free to join now.  
If you’d like to receive a certificate, you’ll need to submit your project while we’re still accepting submissions.


In [23]:
def llm(instructions, prompt, model="openai/gpt-oss-20b"):
    message_history = [
        {"role": 'developer', 'content': instructions},
        {"role": 'user', 'content': prompt}
    ]
    response = openai_client.responses.create(
        model=model,
        input=message_history
    )
    return response.output_text


In [24]:
def rag(query, model = "openai/gpt-oss-20b"):
    # 1. Retrieve the most relevant documents
    search_result = search(query)

    # 2. Generate the prompt with the most relevant documents
    prompt = build_prompt(query, search_result)
    
    # 3. Augment the context with the llm
    answer = llm(INSTRUCTIONS, prompt, model=model)

    return answer
    

In [26]:
answer = rag(question)
print(question)
print(answer)

I just discovered the course. Can I join now?
Yes—feel free to join! Just keep in mind that if you’d like to receive a certificate, you’ll need to submit your project while submissions are still open.
